# Task 5 — Geographic Pre-Clustering

Run unsupervised clustering on L8 basins using Band A+B+C scalar variables (post-F4.9 reductions: 22 variables). Produce a global map of cluster assignments, cluster summary statistics, and comparison with the existing workbench 20-cluster result (`cluster_id` column in basin08).

**Variable selection**: Bands A+B+C only (37 total → 22 after dropping redundant pairs identified in Task 4).

**Normalization**: log1p for non-negative right-skewed variables; StandardScaler throughout. Temperature (can be negative after ×0.1 scale) gets StandardScaler only. Nulls imputed with column median before scaling.

**Methods**: k-means (k=20) and HDBSCAN — run both, compare. k-means matches the existing workbench cluster count for direct comparison.

See `docs/edop/data_exploration.md` Task 5 and findings F4.1–F4.9 for context.

In [ ]:
# Cell 1 — Imports and connection
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, HDBSCAN
from sklearn.metrics import adjusted_rand_score
import os, sys

sys.path.insert(0, '/Users/karlg/Documents/Repos/_cedop')
from scripts.shared.db_utils import db_connect

conn = db_connect()
print("connected")

In [ ]:
# Cell 2 — Variable selection: Bands A+B+C, post-F4.9 reductions
#
# Dropped from full set:
#   slope_upstream (r=0.851 with slope_avg — below 0.9 threshold but high enough
#                   to add limited new information; retained in Task 4 matrix)
#   discharge_min, discharge_max (r>0.93 with discharge_yr)
#   river_area_upstream (r=0.937 with discharge_max)
#   pct_sand (constrained sum: clay+silt+sand≈100; two of three are sufficient)
#   temp_yr_upstream, precip_yr_upstream, aridity_upstream (r>0.984 with local vars)
#   karst_upstream (r=0.807 with karst — below threshold, but included as a
#                   judgment call to keep set lean; reverting: include both)
#
# Each entry: (api_key, basin08_col, log1p_transform, scale_factor, band)
#   log1p: True for non-negative right-skewed variables
#   scale:  0.1 for temperature (stored ×10), 1.0 otherwise

CLUSTER_VARS = [
    # Band A — Terrain / Geology
    ('elev_min',         'ele_mt_smn', True,  1.0, 'A'),
    ('elev_max',         'ele_mt_smx', True,  1.0, 'A'),
    ('slope_avg',        'slp_dg_sav', True,  1.0, 'A'),
    ('stream_gradient',  'sgr_dk_sav', True,  1.0, 'A'),
    ('karst',            'kar_pc_sse', True,  1.0, 'A'),
    # Band B — Hydrology / Soils
    ('discharge_yr',     'dis_m3_pyr', True,  1.0, 'B'),
    ('runoff',           'run_mm_syr', True,  1.0, 'B'),
    ('river_area',       'ria_ha_ssu', True,  1.0, 'B'),
    ('gw_table_depth',   'gwt_cm_sav', True,  1.0, 'B'),
    ('pct_clay',         'cly_pc_sav', False, 1.0, 'B'),
    ('pct_silt',         'slt_pc_sav', False, 1.0, 'B'),
    ('wet_pct_grp1',     'wet_pc_sg1', True,  1.0, 'B'),
    ('wet_pct_grp2',     'wet_pc_sg2', True,  1.0, 'B'),
    ('reservoir_vol',    'rev_mc_usu', True,  1.0, 'B'),
    # Band C — Climate
    ('temp_yr',          'tmp_dc_syr', False, 0.1, 'C'),
    ('temp_min',         'tmp_dc_smn', False, 0.1, 'C'),
    ('temp_max',         'tmp_dc_smx', False, 0.1, 'C'),
    ('precip_yr',        'pre_mm_syr', True,  1.0, 'C'),
    ('aridity',          'ari_ix_sav', True,  1.0, 'C'),
    ('permafrost_extent','prm_pc_sse', True,  1.0, 'C'),
]

api_keys = [v[0] for v in CLUSTER_VARS]
db_cols  = [v[1] for v in CLUSTER_VARS]
print(f"{len(CLUSTER_VARS)} variables: "
      f"A={sum(1 for v in CLUSTER_VARS if v[4]=='A')} "
      f"B={sum(1 for v in CLUSTER_VARS if v[4]=='B')} "
      f"C={sum(1 for v in CLUSTER_VARS if v[4]=='C')}")

In [ ]:
# Cell 3 — Fetch basin data: clustering variables + hybas_id + centroids + existing cluster_id

col_sql = ', '.join(f'"{c}"' for c in db_cols)
query = f"""
    SELECT hybas_id,
           ST_X(ST_Centroid(geom)) AS lon,
           ST_Y(ST_Centroid(geom)) AS lat,
           cluster_id,
           {col_sql}
    FROM public.basin08
"""
with conn.cursor() as cur:
    cur.execute(query)
    rows = cur.fetchall()
    fetch_cols = ['hybas_id', 'lon', 'lat', 'cluster_id'] + db_cols
    df_raw = pd.DataFrame(rows, columns=fetch_cols)

df_raw = df_raw.replace(-9999, np.nan)
print(f"Loaded {len(df_raw):,} rows")
print(f"Existing cluster_id null: {df_raw.cluster_id.isna().sum()}")

In [ ]:
# Cell 4 — Apply scale factors, log1p transforms, median imputation, and StandardScaler

df_feat = pd.DataFrame(index=df_raw.index)

for api_key, db_col, do_log1p, scale, band in CLUSTER_VARS:
    col = df_raw[db_col] * scale
    if do_log1p:
        # log1p(x): handles zeros cleanly; requires x >= 0
        # clamp negatives to 0 (should not occur for non-temp variables)
        col = np.log1p(col.clip(lower=0))
    df_feat[api_key] = col

# Median imputation for nulls
null_counts = df_feat.isnull().sum()
print("Null counts before imputation (non-zero only):")
print(null_counts[null_counts > 0])

for col in df_feat.columns:
    med = df_feat[col].median()
    df_feat[col] = df_feat[col].fillna(med)

# StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(df_feat.values)
print(f"\nFeature matrix shape: {X.shape}")
print(f"Mean ≈ 0: {X.mean(axis=0).max():.4f} (should be ~0)")
print(f"Std  ≈ 1: {X.std(axis=0).mean():.4f} (should be ~1)")

In [ ]:
# Cell 5 — k-means clustering (k=20)
#
# k=20 matches the existing workbench cluster count for direct comparison.
# n_init=10: run 10 times with different seeds, keep best inertia.
# random_state=42 for reproducibility.

print("Running k-means (k=20, n_init=10)...")
km = KMeans(n_clusters=20, n_init=10, random_state=42, verbose=0)
km_labels = km.fit_predict(X)
df_raw['km_cluster'] = km_labels

print(f"Done. Inertia: {km.inertia_:,.0f}")
print("\nCluster sizes:")
sizes = pd.Series(km_labels).value_counts().sort_index()
print(sizes.to_string())

In [ ]:
# Cell 6 — HDBSCAN clustering
#
# min_cluster_size=1000: a coherent environmental type must contain at least
# 1000 of the 190k basins (~0.5%). Smaller clusters are noise.
# min_samples=50: a core point must have 50 neighbors.
#
# NOTE: HDBSCAN on 190k × 20 dimensions may take 10–30 minutes.
# Cluster label -1 = noise (unclustered).

print("Running HDBSCAN (min_cluster_size=1000)... this may take a while.")
hdb = HDBSCAN(min_cluster_size=1000, min_samples=50, n_jobs=-1)
hdb_labels = hdb.fit_predict(X)
df_raw['hdb_cluster'] = hdb_labels

n_clusters = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
n_noise    = (hdb_labels == -1).sum()
print(f"\nHDBSCAN: {n_clusters} clusters, {n_noise:,} noise points "
      f"({n_noise/len(hdb_labels)*100:.1f}%)")
print("\nHDBSCAN cluster sizes (excl. noise):")
hdb_sizes = pd.Series(hdb_labels[hdb_labels >= 0]).value_counts().sort_index()
print(hdb_sizes.to_string())

In [ ]:
# Cell 7 — Global map: k-means cluster assignments
#
# Basin centroids colored by cluster. No basemap — the distribution of
# 190k points is dense enough to define continental outlines.

out_dir = '/Users/karlg/Documents/Repos/_cedop/output/edop/explore'
os.makedirs(out_dir, exist_ok=True)

cmap = plt.get_cmap('tab20', 20)

fig, ax = plt.subplots(figsize=(18, 9))
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')

for c in range(20):
    mask = df_raw['km_cluster'] == c
    ax.scatter(df_raw.loc[mask, 'lon'], df_raw.loc[mask, 'lat'],
               s=0.3, color=cmap(c), alpha=0.6, linewidths=0,
               label=f'C{c} (n={mask.sum():,})')

ax.set_xlim(-180, 180)
ax.set_ylim(-60, 85)
ax.set_title('k-means clusters (k=20) — Band A+B+C variables, L8 basins',
             color='white', fontsize=12)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('gray')

legend = ax.legend(loc='lower left', fontsize=5.5, ncol=4,
                   framealpha=0.4, labelcolor='white',
                   facecolor='#1a1a2e', edgecolor='gray')

plt.tight_layout()
plt.savefig(f'{out_dir}/05_kmeans_global_map.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("Saved 05_kmeans_global_map.png")

In [ ]:
# Cell 8 — Global map: HDBSCAN cluster assignments

n_hdb = len(set(hdb_labels)) - (1 if -1 in hdb_labels else 0)
cmap_hdb = plt.get_cmap('tab20', max(n_hdb, 1))

fig, ax = plt.subplots(figsize=(18, 9))
ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')

# Plot noise in gray first
noise_mask = df_raw['hdb_cluster'] == -1
ax.scatter(df_raw.loc[noise_mask, 'lon'], df_raw.loc[noise_mask, 'lat'],
           s=0.2, color='gray', alpha=0.3, linewidths=0, label=f'noise ({noise_mask.sum():,})')

for c in sorted(set(hdb_labels[hdb_labels >= 0])):
    mask = df_raw['hdb_cluster'] == c
    ax.scatter(df_raw.loc[mask, 'lon'], df_raw.loc[mask, 'lat'],
               s=0.4, color=cmap_hdb(c % n_hdb), alpha=0.7, linewidths=0,
               label=f'C{c} ({mask.sum():,})')

ax.set_xlim(-180, 180)
ax.set_ylim(-60, 85)
ax.set_title(f'HDBSCAN clusters ({n_hdb} clusters) — Band A+B+C variables, L8 basins',
             color='white', fontsize=12)
ax.tick_params(colors='white')
for spine in ax.spines.values():
    spine.set_edgecolor('gray')

ax.legend(loc='lower left', fontsize=5.5, ncol=4,
          framealpha=0.4, labelcolor='white',
          facecolor='#1a1a2e', edgecolor='gray')

plt.tight_layout()
plt.savefig(f'{out_dir}/05_hdbscan_global_map.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print("Saved 05_hdbscan_global_map.png")

In [ ]:
# Cell 9 — k-means cluster summary statistics
#
# Mean of each input variable per cluster, in original (unscaled) units.
# Sorted by mean annual temperature (temp_yr) to give clusters
# an intuitive cold→warm ordering.

# Merge cluster labels back with original (unscaled) feature values
df_feat_orig = pd.DataFrame(index=df_raw.index)
for api_key, db_col, do_log1p, scale, band in CLUSTER_VARS:
    df_feat_orig[api_key] = df_raw[db_col] * scale  # scaled units, no log

df_feat_orig['km_cluster'] = df_raw['km_cluster']

cluster_summary = df_feat_orig.groupby('km_cluster').mean().round(2)
cluster_summary['n_basins'] = df_feat_orig.groupby('km_cluster').size()

# Sort by temp_yr (cold → warm)
cluster_summary = cluster_summary.sort_values('temp_yr')
cluster_summary.insert(0, 'n_basins', cluster_summary.pop('n_basins'))

print("k-means cluster summary (sorted cold → warm):")
print(cluster_summary[['n_basins','temp_yr','precip_yr','aridity','slope_avg',
                        'discharge_yr','permafrost_extent']].to_string())

cluster_summary.to_csv(f'{out_dir}/05_kmeans_cluster_summary.csv')
print("\nSaved 05_kmeans_cluster_summary.csv")

In [ ]:
# Cell 10 — Comparison with existing workbench cluster_id
#
# Adjusted Rand Index (ARI): measures agreement between two clusterings
# corrected for chance. ARI=1: perfect agreement. ARI=0: chance-level.
# ARI<0: worse than chance.
#
# Also show a contingency heatmap: new k-means vs old cluster_id.
# Each cell = fraction of old-cluster basins assigned to each new cluster.

valid = df_raw['cluster_id'].notna()
ari = adjusted_rand_score(df_raw.loc[valid, 'cluster_id'].astype(int),
                          df_raw.loc[valid, 'km_cluster'])
print(f"Adjusted Rand Index (new k-means vs existing cluster_id): {ari:.4f}")
print("(1.0 = identical, 0.0 = chance-level agreement)")

# Contingency table: rows = existing cluster_id, cols = new km_cluster
contingency = pd.crosstab(
    df_raw.loc[valid, 'cluster_id'].astype(int),
    df_raw.loc[valid, 'km_cluster'],
    normalize='index'
).round(2)

fig, ax = plt.subplots(figsize=(12, 9))
im = ax.imshow(contingency.values, cmap='Blues', vmin=0, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax, label='Fraction of old-cluster basins')
ax.set_xticks(range(20))
ax.set_yticks(range(20))
ax.set_xticklabels([f'new {c}' for c in contingency.columns], rotation=90, fontsize=7)
ax.set_yticklabels([f'old {c}' for c in contingency.index], fontsize=7)
ax.set_xlabel('New k-means cluster', fontsize=9)
ax.set_ylabel('Existing workbench cluster_id', fontsize=9)
ax.set_title(f'Cluster correspondence: new k-means vs existing (ARI={ari:.3f})',
             fontsize=10)
plt.tight_layout()
plt.savefig(f'{out_dir}/05_cluster_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved 05_cluster_comparison.png")

# Save assignments
df_raw[['hybas_id', 'km_cluster', 'hdb_cluster', 'cluster_id']].to_csv(
    f'{out_dir}/05_cluster_assignments.csv', index=False)
print("Saved 05_cluster_assignments.csv")